In [2]:
!pip install fuzzywuzzy

In [3]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import pandas as pd
import numpy as np
from collections import OrderedDict
import scipy.stats as stats

/usr/local/lib/python3.12/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [4]:
def extractLastNames(names):
    lastNames = []
    for name in names:
      parts = name.strip().lower().split()
      lastNames.append(parts[-1].strip())
    return lastNames

In [5]:
thresholds = [60, 70, 80, 90]

In [6]:
df = pd.read_csv("researchers.csv")
df.head()

,Name,Google Scholar ID,OpenAlex URL,Citation Count,Publication Count,Email Domain,Affiliation,Field,Subfield,Citation Level,...,Region,Co-author count (Google Scholar),Co-authors' names (Google Scholar),Co-authors' names (OpenAlex),Co-authors' names (DeepSeek R1),Reality (DeepSeek R1),Co-authors' names (Llama 4 Scout),Reality (Llama 4 Scout),Co-authors' names (Mixtral),Reality (Mixtral)
0,Kuttanelloor Roshni,xvwC5SMAAAAJ,https://openalex.org/a5049943816,163,29,@cusat.ac.in,"Post Doctoral Fellow, School of Environmental ...","Agriculture, Fisheries & Forestry",Fisheries,Low,...,East/South-East Asia,3,K Ranjeet/Neelesh Dahanukar/Rajeev Raghavan,B Madhusoodana Kurup/Balakrishna Madhusoodana ...,Shashi Bhushan/Vinay Kumar Vase/Achamveetil Go...,-,NaN,-,S. Sree Ranjani/S. Jegadhesan/R. Anbazhagan/K ...,-
1,Siriluck Thammanu,H6zh8jwAAAAJ,https://openalex.org/A5090976916,120,8,@mail.forest.go.th,"Royal Forest Department, Thailand","Agriculture, Fisheries & Forestry",Forestry,Low,...,East/South-East Asia,3,Jamroon Srichaichana/Dokrak Marod/Hee Han,Akbar I. Inamdar/Arjun Magotra/Atchara Teerawa...,Harald Vacik/Eakaphong Ashraf/Shinya Numata,-,NaN,-,Pattarapong Kongcharoen/Kitiyos Srisuksanti/Su...,-
2,M.S.M. Nafees,eQr4PLYAAAAJ,https://openalex.org/a5079244643,113,15,@esn.ac.lk,Senior Lecturer,"Agriculture, Fisheries & Forestry",Dairy & Animal Science,Low,...,East/South-East Asia,3,A.R.S.B. Athauda/Saman Athauda/M Pagthinathan/...,A. R. S. B. Athauda/Clement R. de Cruz/E. Pows...,Qamar Bilal/Aisha Khatoon/Muhammad Tahir,-,NaN,-,Muhammad Farrukh Shahid/Abdul Rauf/Muhammad Im...,-
3,Aman Kumar Gupta,cab6DuAAAAAJ,https://openalex.org/a5101497736,122,17,NaN,Banaras Hindu University,"Agriculture, Fisheries & Forestry",Agronomy & Agriculture,Low,...,East/South-East Asia,3,Poulomi Dey/Jiwan Paudel/Ashish Chaudhary,Akshay Chaudhary/Alberto Sanz-Cobeña/Alice Min...,Rajendra Prasad/Anil Kumar Singh/Shiv Datt Sharma,-,NaN,-,Surjeet Singh/Jaspreet Singh/Amrik S. Basra,-
4,Seralathan Kamala‐Kannan,ERXuIzUAAAAJ,https://openalex.org/A5075977093,158,178,@annamalaiuniversity.ac.in,"Department of Horticulture, Faculty of Agricul...","Agriculture, Fisheries & Forestry",Horticulture,Low,...,East/South-East Asia,5,S. Venkatesan/T.Uma Maheswari/S Kumar/R. Sudha...,A. Bedoui/A. Deepak/A. K. Ramasamy/A. Kanakala...,K. Prabakar/R. Samiyappan/A. Nithya/M. Senthil...,-,NaN,-,Ramalingam Palanisamy/Arulmozhi Marimuthu/Siva...,-


In [7]:
df = df[df["Reality (Mixtral)"] != "hypothetical"]
df = df[df["Co-authors' names (Mixtral)"].notna()]

In [8]:
len(df)

1518

In [9]:
df["Match Count 60%"] = 0
df["Match Count 70%"] = 0
df["Match Count 80%"] = 0
df["Match Count 90%"] = 0

In [10]:
for index, row in df.iterrows():
    llmNames = row["Co-authors' names (Mixtral)"].split('/')
    refNames = row["Co-authors' names (Google Scholar)"].split('/')

    llmNames = extractLastNames(llmNames)
    refNames = extractLastNames(refNames)

    for threshold in thresholds:
        count = 0
        refNamesCopy = refNames.copy()

        for name in llmNames:
            if not refNamesCopy:
                break

            bestRatio = 0
            bestIndex = -1
            for i, candidate in enumerate(refNamesCopy):
                currentRatio = fuzz.ratio(name, candidate)
                if currentRatio >= threshold and currentRatio > bestRatio:
                    bestRatio = currentRatio
                    bestIndex = i

            if bestIndex != -1:
                count += 1
                del refNamesCopy[bestIndex]

        colName = "Match Count " + str(threshold) + "%"
        df.at[index, colName] = count

In [11]:
df.head()

,Name,Google Scholar ID,OpenAlex URL,Citation Count,Publication Count,Email Domain,Affiliation,Field,Subfield,Citation Level,...,Co-authors' names (DeepSeek R1),Reality (DeepSeek R1),Co-authors' names (Llama 4 Scout),Reality (Llama 4 Scout),Co-authors' names (Mixtral),Reality (Mixtral),Match Count 60%,Match Count 70%,Match Count 80%,Match Count 90%
0,Kuttanelloor Roshni,xvwC5SMAAAAJ,https://openalex.org/a5049943816,163,29,@cusat.ac.in,"Post Doctoral Fellow, School of Environmental ...","Agriculture, Fisheries & Forestry",Fisheries,Low,...,Shashi Bhushan/Vinay Kumar Vase/Achamveetil Go...,-,NaN,-,S. Sree Ranjani/S. Jegadhesan/R. Anbazhagan/K ...,-,1,1,1,1
1,Siriluck Thammanu,H6zh8jwAAAAJ,https://openalex.org/A5090976916,120,8,@mail.forest.go.th,"Royal Forest Department, Thailand","Agriculture, Fisheries & Forestry",Forestry,Low,...,Harald Vacik/Eakaphong Ashraf/Shinya Numata,-,NaN,-,Pattarapong Kongcharoen/Kitiyos Srisuksanti/Su...,-,0,0,0,0
2,M.S.M. Nafees,eQr4PLYAAAAJ,https://openalex.org/a5079244643,113,15,@esn.ac.lk,Senior Lecturer,"Agriculture, Fisheries & Forestry",Dairy & Animal Science,Low,...,Qamar Bilal/Aisha Khatoon/Muhammad Tahir,-,NaN,-,Muhammad Farrukh Shahid/Abdul Rauf/Muhammad Im...,-,0,0,0,0
3,Aman Kumar Gupta,cab6DuAAAAAJ,https://openalex.org/a5101497736,122,17,NaN,Banaras Hindu University,"Agriculture, Fisheries & Forestry",Agronomy & Agriculture,Low,...,Rajendra Prasad/Anil Kumar Singh/Shiv Datt Sharma,-,NaN,-,Surjeet Singh/Jaspreet Singh/Amrik S. Basra,-,0,0,0,0
4,Seralathan Kamala‐Kannan,ERXuIzUAAAAJ,https://openalex.org/A5075977093,158,178,@annamalaiuniversity.ac.in,"Department of Horticulture, Faculty of Agricul...","Agriculture, Fisheries & Forestry",Horticulture,Low,...,K. Prabakar/R. Samiyappan/A. Nithya/M. Senthil...,-,NaN,-,Ramalingam Palanisamy/Arulmozhi Marimuthu/Siva...,-,0,0,0,0


In [12]:
df["DNE 60%"] = 0
df["DNE 70%"] = 0
df["DNE 80%"] = 0
df["DNE 90%"] = 0

for index, row in df.iterrows():
  df.at[index, "DNE 60%"] = row["Match Count 60%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
  df.at[index, "DNE 70%"] = row["Match Count 70%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
  df.at[index, "DNE 80%"] = row["Match Count 80%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
  df.at[index, "DNE 90%"] = row["Match Count 90%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))

/tmp/ipython-input-558499341.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3333333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, "DNE 60%"] = row["Match Count 60%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
/tmp/ipython-input-558499341.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3333333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, "DNE 70%"] = row["Match Count 70%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
/tmp/ipython-input-558499341.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3333333333333333' has dtype incompatible with int64, please explicitly

In [13]:
print("Mean DNE", np.mean(df["DNE 60%"]))
print("Std DNE", df["DNE 60%"].std())

Mean DNE 0.11679043044419965
Std DNE 0.1642877585945751


In [14]:
fields = list(df["Field"].unique())
fields

['Agriculture, Fisheries & Forestry',
 'Biology',
 'Built Environment & Design',
 'Clinical Medicine',
 'Earth & Environmental Sciences',
 'Economics & Business',
 'Engineering',
 'Information & Communication Technologies',
 'Mathematics & Statistics',
 'Physics & Astronomy']

In [15]:
metrics = ["DNE 60%", "DNE 70%", "DNE 80%", "DNE 90%"]

levels = ['High', 'Low']

In [16]:
regions = list(df["Region"].unique())
regions

['East/South-East Asia',
 'Europe',
 'Middle East',
 'North Africa',
 'North America',
 'Oceanic',
 'South/Central America',
 'Sub-Saharan Africa']

# Mean DNE without disaggregation

In [17]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
      print(metric, "*****", sf[metric].mean())

Agriculture, Fisheries & Forestry
DNE 60% ***** 0.11111972370269964
DNE 70% ***** 0.050658319394767
DNE 80% ***** 0.0368090235881447
DNE 90% ***** 0.032437449993937134
Biology
DNE 60% ***** 0.12211971897188328
DNE 70% ***** 0.04590503006529826
DNE 80% ***** 0.030948036904069134
DNE 90% ***** 0.02369026157483781
Built Environment & Design
DNE 60% ***** 0.10145291640727586
DNE 70% ***** 0.04377600089348206
DNE 80% ***** 0.027676433751919263
DNE 90% ***** 0.01751864060655124
Clinical Medicine
DNE 60% ***** 0.11083586954674902
DNE 70% ***** 0.053843330643259836
DNE 80% ***** 0.03636478175209612
DNE 90% ***** 0.02887974499520676
Earth & Environmental Sciences
DNE 60% ***** 0.1352804706909255
DNE 70% ***** 0.05759694915600098
DNE 80% ***** 0.03495661477633575
DNE 90% ***** 0.019665698368450512
Economics & Business
DNE 60% ***** 0.08976569896606228
DNE 70% ***** 0.03474988510450523
DNE 80% ***** 0.022800707360366036
DNE 90% ***** 0.017165601846081107
Engineering
DNE 60% ***** 0.11838080201314

In [18]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    print(metric, "*****", sf[metric].mean())

East/South-East Asia
DNE 60% ***** 0.17984798767594104
DNE 70% ***** 0.09224049466639597
DNE 80% ***** 0.07494508428444856
DNE 90% ***** 0.05422972206311315
Europe
DNE 60% ***** 0.09451452282092031
DNE 70% ***** 0.04156822062112442
DNE 80% ***** 0.02319704694820885
DNE 90% ***** 0.018450856436875423
Middle East
DNE 60% ***** 0.1386822179981381
DNE 70% ***** 0.04931324519764927
DNE 80% ***** 0.02483924354277282
DNE 90% ***** 0.014331089688459865
North Africa
DNE 60% ***** 0.12492969713235512
DNE 70% ***** 0.04422498927925244
DNE 80% ***** 0.0181925576637997
DNE 90% ***** 0.010681959309071494
North America
DNE 60% ***** 0.10567166443326226
DNE 70% ***** 0.04892025850412596
DNE 80% ***** 0.0368280810903838
DNE 90% ***** 0.03012775785130209
Oceanic
DNE 60% ***** 0.11317537709823344
DNE 70% ***** 0.05232900421989703
DNE 80% ***** 0.03530358922347733
DNE 90% ***** 0.022388143290193677
South/Central America
DNE 60% ***** 0.08530958334787256
DNE 70% ***** 0.039596359615895894
DNE 80% ***** 0.0

# Mean DNE disaggregated by Citation Level

In [19]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for level in levels:
    print(level)
    hf = sf[sf["Citation Level"] == level]
    for metric in metrics:
      print(metric, "*****", hf[metric].mean())

Agriculture, Fisheries & Forestry
High
DNE 60% ***** 0.13587734956590414
DNE 70% ***** 0.06758239889642739
DNE 80% ***** 0.05322993624860297
DNE 90% ***** 0.04543341276895987
Low
DNE 60% ***** 0.08636209783949511
DNE 70% ***** 0.0337342398931066
DNE 80% ***** 0.02038811092768644
DNE 90% ***** 0.019441487218914394
Biology
High
DNE 60% ***** 0.1516563115004221
DNE 70% ***** 0.06610129080500435
DNE 80% ***** 0.04507228588434635
DNE 90% ***** 0.03408360923466319
Low
DNE 60% ***** 0.09177390473023378
DNE 70% ***** 0.02515544711354542
DNE 80% ***** 0.016436822198304876
DNE 90% ***** 0.013012164664058296
Built Environment & Design
High
DNE 60% ***** 0.13071455622260167
DNE 70% ***** 0.06159154561134422
DNE 80% ***** 0.04018873927094114
DNE 90% ***** 0.02904604832366259
Low
DNE 60% ***** 0.0709887434488544
DNE 70% ***** 0.025228310502283104
DNE 80% ***** 0.014649923896499238
DNE 90% ***** 0.005517503805175038
Clinical Medicine
High
DNE 60% ***** 0.13480714750546868
DNE 70% ***** 0.066679030110

In [20]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for level in levels:
    print(level)
    hf = sf[sf["Citation Level"] == level]
    for metric in metrics:
      print(metric, "*****", hf[metric].mean())

East/South-East Asia
High
DNE 60% ***** 0.23419127679435056
DNE 70% ***** 0.13652849755449076
DNE 80% ***** 0.11317358273233115
DNE 90% ***** 0.08982986282997651
Low
DNE 60% ***** 0.12375168923113117
DNE 70% ***** 0.04652384652384652
DNE 80% ***** 0.03548340846727943
DNE 90% ***** 0.017481189658609016
Europe
High
DNE 60% ***** 0.12460858273374457
DNE 70% ***** 0.05718621963888978
DNE 80% ***** 0.030054335420496733
DNE 90% ***** 0.022545262931406268
Low
DNE 60% ***** 0.06276694313266613
DNE 70% ***** 0.025092089789196123
DNE 80% ***** 0.015962984384037017
DNE 90% ***** 0.014131482552535183
Middle East
High
DNE 60% ***** 0.1697312305612435
DNE 70% ***** 0.06191791131582913
DNE 80% ***** 0.031270921669979974
DNE 90% ***** 0.018745193362428483
Low
DNE 60% ***** 0.10731311252201105
DNE 70% ***** 0.03657863406794178
DNE 80% ***** 0.018341259455491365
DNE 90% ***** 0.00987147979104827
North Africa
High
DNE 60% ***** 0.1507169118837019
DNE 70% ***** 0.058032317499584374
DNE 80% ***** 0.0289817

# Mean Comparison

In [21]:
lowCitedDNE = df[df["Citation Level"] == "Low"]["DNE 60%"].to_list()
highlyCitedDNE = df[df["Citation Level"] == "High"]["DNE 60%"].to_list()

In [22]:
print("Low cited Avg DNE:", np.mean(lowCitedDNE))
print("highly cited Avg DNE:", np.mean(highlyCitedDNE))

Low cited Avg DNE: 0.08123847962308028
highly cited Avg DNE: 0.15187704159488552


In [23]:
t_stat, p_value = stats.ttest_ind(highlyCitedDNE, lowCitedDNE, alternative='greater')

In [24]:
print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

T-statistic: 8.573771250529088
P-value: 1.2155385450314785e-17


## T-Test

In [25]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    t_stat, p_value = stats.ttest_ind(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

Agriculture, Fisheries & Forestry
DNE 60% ***** 0.024418231730450664
DNE 70% ***** 0.032196693872923024
DNE 80% ***** 0.026456668851566504
DNE 90% ***** 0.05997913931133266
Biology
DNE 60% ***** 0.010061528492401899
DNE 70% ***** 0.0003843848382212457
DNE 80% ***** 0.0023071884668520153
DNE 90% ***** 0.006561328350883089
Built Environment & Design
DNE 60% ***** 0.006801682266578302
DNE 70% ***** 0.008126113300356988
DNE 80% ***** 0.015660162470058206
DNE 90% ***** 0.005679213558854468
Clinical Medicine
DNE 60% ***** 0.035769898943875805
DNE 70% ***** 0.09061270831514047
DNE 80% ***** 0.23771064447055507
DNE 90% ***** 0.32777532651982744
Earth & Environmental Sciences
DNE 60% ***** 0.002173932165406713
DNE 70% ***** 0.07184001975573633
DNE 80% ***** 0.2965242853382185
DNE 90% ***** 0.01880214138070601
Economics & Business
DNE 60% ***** 0.00011846021849443956
DNE 70% ***** 0.00010367142730759063
DNE 80% ***** 0.0014258751575097971
DNE 90% ***** 0.015980057437886355
Engineering
DNE 60% **

In [26]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    t_stat, p_value = stats.ttest_ind(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

East/South-East Asia
DNE 60% ***** 0.0004105004354456527
DNE 70% ***** 0.0001240064831624552
DNE 80% ***** 0.00044518375679282793
DNE 90% ***** 7.950370808250442e-05
Europe
DNE 60% ***** 0.0004903702137072742
DNE 70% ***** 0.013773989722519041
DNE 80% ***** 0.059923105407026216
DNE 90% ***** 0.1629289724464893
Middle East
DNE 60% ***** 0.009303647715209492
DNE 70% ***** 0.027280032582947342
DNE 80% ***** 0.08894698656832568
DNE 90% ***** 0.12254961440417332
North Africa
DNE 60% ***** 0.02652558786420288
DNE 70% ***** 0.03297145273753451
DNE 80% ***** 9.386852242135423e-05
DNE 90% ***** 0.002449903672180534
North America
DNE 60% ***** 1.531509905535058e-05
DNE 70% ***** 0.0045783218839267545
DNE 80% ***** 0.007308194991845965
DNE 90% ***** 0.010134945568107415
Oceanic
DNE 60% ***** 0.002503992896839946
DNE 70% ***** 0.0024724440952339803
DNE 80% ***** 0.002064862765120512
DNE 90% ***** 0.0038332631108856543
South/Central America
DNE 60% ***** 7.30801262561055e-09
DNE 70% ***** 4.6225355

## Mann-Whitney U Test

In [27]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    u_stat, p_value = stats.mannwhitneyu(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

Agriculture, Fisheries & Forestry
DNE 60% ***** 0.0060327831265835795
DNE 70% ***** 0.005203682389209056
DNE 80% ***** 0.0002508095451724778
DNE 90% ***** 0.0016448350817117384
Biology
DNE 60% ***** 1.3112823730169677e-05
DNE 70% ***** 3.3302528559734617e-07
DNE 80% ***** 6.34356118131099e-06
DNE 90% ***** 0.00011832138368383735
Built Environment & Design
DNE 60% ***** 6.49125159493929e-05
DNE 70% ***** 1.4489379570433469e-05
DNE 80% ***** 1.1354906117180603e-05
DNE 90% ***** 7.667801764178816e-05
Clinical Medicine
DNE 60% ***** 0.0024316202643105238
DNE 70% ***** 0.01614575702434069
DNE 80% ***** 0.07211995705932031
DNE 90% ***** 0.07444097216809062
Earth & Environmental Sciences
DNE 60% ***** 7.324120243336488e-05
DNE 70% ***** 1.865727885077178e-05
DNE 80% ***** 0.00010678499754417571
DNE 90% ***** 8.34752879804672e-05
Economics & Business
DNE 60% ***** 5.093001501589997e-08
DNE 70% ***** 6.890506424690956e-07
DNE 80% ***** 5.572915320383916e-06
DNE 90% ***** 0.00020921768518724216


In [28]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    u_stat, p_value = stats.mannwhitneyu(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

East/South-East Asia
DNE 60% ***** 1.194513038322485e-05
DNE 70% ***** 1.6314247180668786e-06
DNE 80% ***** 2.77463217029006e-06
DNE 90% ***** 4.396906634115175e-06
Europe
DNE 60% ***** 4.9591596234783964e-05
DNE 70% ***** 4.811049929199635e-06
DNE 80% ***** 3.6511620449000274e-06
DNE 90% ***** 1.87101845930917e-05
Middle East
DNE 60% ***** 6.521925026022581e-06
DNE 70% ***** 9.113782138058955e-05
DNE 80% ***** 1.019865255566632e-05
DNE 90% ***** 0.0014126401499046446
North Africa
DNE 60% ***** 0.0001173714532263632
DNE 70% ***** 2.002276961039223e-06
DNE 80% ***** 1.9398138219663554e-06
DNE 90% ***** 0.00011922316504704617
North America
DNE 60% ***** 7.56493144150892e-08
DNE 70% ***** 5.6457502910857955e-06
DNE 80% ***** 2.4738273226880035e-06
DNE 90% ***** 1.5771268260410913e-05
Oceanic
DNE 60% ***** 2.9327287705694817e-06
DNE 70% ***** 1.68628550755455e-06
DNE 80% ***** 9.85872782854618e-07
DNE 90% ***** 5.58260251843358e-06
South/Central America
DNE 60% ***** 4.276474798509722e-10
